v4 — ensemble LightGBM + CatBoost, multi-day CV, Optuna
P@R70 ≈ 0.73 по CV

In [1]:
import re
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from catboost import CatBoostClassifier
from user_agents import parse as ua_parse
from metric import precision_at_recall

optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42
CAT_COLS = ["ua_top_browser", "ua_top_os"]

C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## загрузка и чистка

In [2]:
def load_events():
    train = pd.read_csv("data/train.csv", parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
    test  = pd.read_csv("data/test.csv",  parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
    meta  = pd.concat([train.drop(columns=["target"]), test], ignore_index=True)

    ev = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"]).drop_duplicates()
    ev["platform"] = ev["platform"].str.lower().str.strip()
    ev = ev.sort_values(["cookie_id", "event_ts"]).reset_index(drop=True)

    ev = ev.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id", how="inner")
    ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]
    return train, test, ev.drop(columns=["window_start_ts", "window_end_ts", "eid"], errors="ignore")


def extract_ua(ev):
    BOT_MARKERS = ["headlesschrome", "scrapy", "python-requests", "python-urllib",
                   "curl", "okhttp", "go-http", "java/", "apache-httpclient"]
    cache = {}
    for s in ev["user_agent"].dropna().unique():
        p = ua_parse(s)
        m = re.search(r"Chrome/(\d+)", s)
        cache[s] = {
            "ua_browser": p.browser.family, "ua_os": p.os.family,
            "ua_is_bot": int(p.is_bot), "ua_is_mobile": int(p.is_mobile),
            "has_bot_marker": int(any(mk in s.lower() for mk in BOT_MARKERS)),
            "chrome_major": int(m.group(1)) if m else np.nan,
        }
    ua_df = pd.DataFrame.from_dict(cache, orient="index").reset_index().rename(columns={"index": "user_agent"})
    ev = ev.merge(ua_df, on="user_agent", how="left")
    # проверка на подставные платформы
    ev["ua_mismatch"] = (
        ((ev["platform"].isin(["android", "ios"])) & (ev["ua_is_mobile"] == 0)) |
        ((ev["platform"] == "desktop") & (ev["ua_is_mobile"] == 1))
    ).astype(int)
    return ev

## признаки

In [3]:
def build_features(ev):
    g = ev.groupby("cookie_id")

    base = pd.DataFrame({
        "n_events": g.size(), "n_unique_items": g["item_id"].nunique(),
        "n_unique_cats": g["item_category"].nunique(),
        "n_unique_locs": g["item_location"].nunique(),
        "n_unique_ua": g["user_agent"].nunique(),
    })

    shares = pd.crosstab(ev["cookie_id"], ev["event_name"], normalize="index").add_prefix("share_")

    # проверкаа интервалов на хаотичность
    ev["ts_diff"] = g["event_ts"].diff().dt.total_seconds()
    timing = g["ts_diff"].agg(interval_mean="mean", interval_std="std",
                               interval_min="min", interval_max="max", interval_median="median")
    timing["interval_cv"] = timing["interval_std"] / (timing["interval_mean"] + 1e-9)
    ev["interval_fast"] = (ev["ts_diff"].abs() < 0.1).astype(float)
    timing["fast_interval_share"] = g["interval_fast"].mean()
    time_span = g["event_ts"].agg(lambda x: (x.max() - x.min()).total_seconds()).rename("session_duration_sec")
    timing["events_per_min"] = base["n_events"] / (time_span / 60 + 1e-9)

    ev["hour"] = ev["event_ts"].dt.hour
    hour_feats = g["hour"].agg(hour_std="std", hour_mean="mean")
    hour_feats["hour_night_share"] = (
        ev.assign(n=ev["hour"].between(0, 6).astype(float)).groupby("cookie_id")["n"].mean()
    )

    # nan у мыши = mobile или скрипт без gui
    # низкий std = бот кликает в одно место
    mouse = g.agg(mouse_x_std=("pointer_x", "std"), mouse_y_std=("pointer_y", "std"),
                  mouse_nan_share=("pointer_x", lambda x: x.isna().mean()),
                  mouse_has_share=("pointer_x", lambda x: x.notna().mean()))

    sg = ev[ev["event_name"] == "search_results_view"].groupby("cookie_id")
    search = sg.agg(search_n_queries=("search_query", "count"), search_unique_q=("search_query", "nunique"),
                    search_max_page=("search_page", "max"), search_mean_page=("search_page", "mean"))
    search["search_query_diversity"] = search["search_unique_q"] / (search["search_n_queries"] + 1e-9)

    # проверка на цикличность событий
    ev["event_code"] = ev["event_name"].astype("category").cat.codes
    ev["prev_event"] = g["event_code"].shift(1)
    ev["event_repeated"] = (ev["event_code"] == ev["prev_event"]).astype(float)
    repeat = g["event_repeated"].agg(event_repeat_share="mean")
    repeat["n_unique_transitions"] = ev.groupby("cookie_id").apply(
        lambda d: len(set(zip(d["event_code"].values[1:], d["event_code"].values[:-1])))
    )

    ivg = ev[ev["event_name"] == "item_view"].groupby("cookie_id")
    item_div = pd.DataFrame({"item_view_n": ivg.size(), "item_view_unique": ivg["item_id"].nunique()})
    item_div["item_view_diversity"] = item_div["item_view_unique"] / (item_div["item_view_n"] + 1e-9)
    n_photo = shares.get("share_photo_swipe", pd.Series(0, index=shares.index)) * base["n_events"]
    n_item  = shares.get("share_item_view",   pd.Series(0, index=shares.index)) * base["n_events"]
    item_div["photo_per_item_view"] = n_photo / (n_item + 1e-9)

    ua_feats = g.agg(ua_bot_share=("ua_is_bot", "mean"), ua_mismatch_share=("ua_mismatch", "mean"),
                     ua_bot_marker_share=("has_bot_marker", "mean"),
                     ua_any_bot_marker=("has_bot_marker", "max"),
                     chrome_major_mean=("chrome_major", "mean"))
    ua_feats["ua_top_browser"] = g["ua_browser"].agg(lambda x: x.value_counts().index[0] if len(x) > 0 else np.nan)
    ua_feats["ua_top_os"]      = g["ua_os"].agg(lambda x: x.value_counts().index[0] if len(x) > 0 else np.nan)
    median_chrome = ev["chrome_major"].median()
    ev["chrome_is_old"] = (ev["chrome_major"].notna() & (ev["chrome_major"] < median_chrome - 10)).astype(float)
    ua_feats["chrome_old_share"] = g["chrome_is_old"].mean()

    seller = ev[ev["seller_type"].notna()].groupby("cookie_id")["seller_type"].apply(
        lambda x: (x == "pro").mean()
    ).rename("seller_pro_share")
    seller_div = ev.groupby("cookie_id")["seller_type"].nunique().rename("seller_type_nunique")

    special = pd.DataFrame({
        "had_captcha": g["event_name"].apply(lambda x: int("captcha_shown" in x.values)),
        "had_login":   g["event_name"].apply(lambda x: int("login" in x.values)),
    })

    h = pd.DataFrame(index=base.index)
    h["h_bot_ua"]     = (ua_feats["ua_any_bot_marker"] > 0).astype(int)
    h["h_fast"]       = (timing["interval_median"] < 1.0).astype(int)
    h["h_regular"]    = (timing["interval_cv"] < 0.3).astype(int)
    h["h_old_chrome"] = (ua_feats["chrome_old_share"] > 0.5).astype(int)
    h["h_mismatch"]   = (ua_feats["ua_mismatch_share"] > 0.5).astype(int)
    h["h_monotone"]   = (repeat["event_repeat_share"] > 0.8).astype(int)
    h["heuristic_score"] = h.sum(axis=1)

    frames = [shares, timing, time_span, hour_feats, mouse, search, repeat,
              item_div, ua_feats, seller, seller_div, special, h]
    feats = base.copy()
    for f in frames:
        if len(f) > 0:
            feats = feats.join(f, how="left")
    return feats.reset_index()


def make_dataset(meta, features):
    df = meta.merge(features, on="cookie_id", how="left")
    df["cookie_age_days"] = ((df["window_start_ts"] - df["cookie_created_at"]).dt.total_seconds() / 86400).clip(lower=0)
    df["cookie_new_in_window"] = (df["cookie_age_days"] < 1).astype(int)
    df["day"] = df["window_start_ts"].dt.date
    return df


def get_feat_cols(df):
    drop = {"cookie_id", "cookie_created_at", "window_start_ts", "window_end_ts", "target", "day"}
    return [c for c in df.columns if c not in drop]

## вспомогательные функции

In [4]:
def prep_lgb(df):
    df = df.copy()
    for c in CAT_COLS:
        if c in df.columns:
            df[c] = df[c].astype("category")
    return df


def prep_cb(df):
    df = df.copy()
    for c in CAT_COLS:
        if c in df.columns:
            df[c] = df[c].fillna("unknown").astype(str)
    return df

## обучение и валидация

In [5]:
def day_cv(df_train, feat_cols, params, n_days=5):
    days = sorted(df_train["day"].unique())[-n_days:]
    scores = []
    for val_day in days:
        tr = df_train[df_train["day"] < val_day]
        va = df_train[df_train["day"] == val_day]
        if va["target"].sum() == 0:
            continue
        m = lgb.LGBMClassifier(**params)
        m.fit(prep_lgb(tr)[feat_cols], tr["target"], eval_set=[(prep_lgb(va)[feat_cols], va["target"])],
              categorical_feature=CAT_COLS,
              callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
        p = m.predict_proba(prep_lgb(va)[feat_cols])[:, 1]
        s = precision_at_recall(va["target"].values, p)
        scores.append(s)
        print(f"  {val_day}: ботов={va['target'].sum():3d}  P@R70={s:.4f}")
    mean = float(np.mean(scores))
    print(f"  среднее: {mean:.4f} ± {np.std(scores):.4f}")
    return mean


def optimize_lgb(df_train, feat_cols, n_trials=50):
    days = sorted(df_train["day"].unique())[-3:]

    def objective(trial):
        params = {
            "objective": "binary", "metric": "auc", "verbose": -1,
            "random_state": RANDOM_STATE, "is_unbalance": True,
            "n_estimators": 500, "bagging_freq": 5,
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "num_leaves":        trial.suggest_int("num_leaves", 31, 200),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
            "feature_fraction":  trial.suggest_float("feature_fraction", 0.5, 1.0),
            "bagging_fraction":  trial.suggest_float("bagging_fraction", 0.5, 1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha",  1e-8, 10.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "min_split_gain":    trial.suggest_float("min_split_gain", 0.0, 1.0),
        }
        scores = []
        for val_day in days:
            tr = df_train[df_train["day"] < val_day]
            va = df_train[df_train["day"] == val_day]
            if va["target"].sum() == 0:
                continue
            m = lgb.LGBMClassifier(**params)
            m.fit(prep_lgb(tr)[feat_cols], tr["target"], eval_set=[(prep_lgb(va)[feat_cols], va["target"])],
                  categorical_feature=CAT_COLS,
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
            p = m.predict_proba(prep_lgb(va)[feat_cols])[:, 1]
            scores.append(precision_at_recall(va["target"].values, p))
        return float(np.mean(scores))

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return {**study.best_params, "objective": "binary", "metric": "auc", "verbose": -1,
            "random_state": RANDOM_STATE, "is_unbalance": True, "n_estimators": 2000, "bagging_freq": 5}


def find_blend_weight(df_train, feat_cols, lgb_params, cb_params):
    days = sorted(df_train["day"].unique())[-3:]
    all_p_lgb, all_p_cb, all_y = [], [], []

    for val_day in days:
        tr = df_train[df_train["day"] < val_day]
        va = df_train[df_train["day"] == val_day]
        if va["target"].sum() == 0:
            continue
        m_lgb = lgb.LGBMClassifier(**lgb_params)
        m_lgb.fit(prep_lgb(tr)[feat_cols], tr["target"], categorical_feature=CAT_COLS,
                  callbacks=[lgb.log_evaluation(-1)])
        m_cb = CatBoostClassifier(**cb_params, verbose=0)
        m_cb.fit(prep_cb(tr)[feat_cols], tr["target"])
        all_p_lgb.append(m_lgb.predict_proba(prep_lgb(va)[feat_cols])[:, 1])
        all_p_cb.append(m_cb.predict_proba(prep_cb(va)[feat_cols])[:, 1])
        all_y.append(va["target"].values)

    p_lgb = np.concatenate(all_p_lgb)
    p_cb = np.concatenate(all_p_cb)
    y = np.concatenate(all_y)

    best_w, best_score = 0.5, 0.0
    for w in np.arange(0.0, 1.01, 0.1):
        s = precision_at_recall(y, w * p_lgb + (1 - w) * p_cb)
        if s > best_score: best_score, best_w = s, w
    return best_w

## запуск

### загрузка данных

In [6]:
train, test, ev = load_events()
ev = extract_ua(ev)

### построение признаков

In [7]:
features = build_features(ev)
df_train = make_dataset(train, features)
df_test = make_dataset(test,  features)
feat_cols = get_feat_cols(df_train)

### baseline cv до optuna

In [8]:
baseline_params = {
    "objective": "binary", "metric": "auc", "verbose": -1,
    "random_state": RANDOM_STATE, "is_unbalance": True,
    "n_estimators": 500, "learning_rate": 0.05, "num_leaves": 63,
    "min_child_samples": 20, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 5,
}
baseline_cv = day_cv(df_train, feat_cols, baseline_params)

C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  2026-04-15: ботов= 65  P@R70=0.6765
  2026-04-16: ботов= 60  P@R70=0.5526


C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  2026-04-17: ботов= 66  P@R70=0.5949
  2026-04-18: ботов= 52  P@R70=0.6271


C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  2026-04-19: ботов= 42  P@R70=0.7692
  среднее: 0.6441 ± 0.0745


### подбор гиперпараметров

In [9]:
best_lgb_params = optimize_lgb(df_train, feat_cols, n_trials=50)

  0%|          | 0/50 [00:00<?, ?it/s]C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 0. Best value: 0.674271:   2%|▏         | 1/50 [00:00<00:39,  1.23it/s]C:\Users\proto\PycharmProjects\Avito_DataCamp

### cv после optuna

In [10]:
optuna_cv = day_cv(df_train, feat_cols, best_lgb_params)

C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  2026-04-15: ботов= 65  P@R70=0.6438
  2026-04-16: ботов= 60  P@R70=0.6000


C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  2026-04-17: ботов= 66  P@R70=0.7581


C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  2026-04-18: ботов= 52  P@R70=0.7255


C:\Users\proto\PycharmProjects\Avito_DataCamp\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  2026-04-19: ботов= 42  P@R70=0.7750
  среднее: 0.7005 ± 0.0675


### подбор весов ансамбля

In [11]:
cb_params = dict(iterations=1000, learning_rate=0.04, depth=6,
                 auto_class_weights="Balanced", cat_features=CAT_COLS,
                 eval_metric="AUC", random_seed=RANDOM_STATE, early_stopping_rounds=50)
blend_w = find_blend_weight(df_train, feat_cols, best_lgb_params, cb_params)

### финальные модели на всём train

In [12]:
lgb_final = lgb.LGBMClassifier(**best_lgb_params)
lgb_final.fit(prep_lgb(df_train)[feat_cols], df_train["target"],
              categorical_feature=CAT_COLS, callbacks=[lgb.log_evaluation(200)])

cb_final = CatBoostClassifier(**cb_params, verbose=0)
cb_final.fit(prep_cb(df_train)[feat_cols], df_train["target"])

imp = pd.Series(lgb_final.feature_importances_, index=feat_cols).sort_values(ascending=False)
print(imp.head(20).to_string())

interval_median              2329
cookie_age_days              1859
hour_mean                    1477
search_mean_page             1415
interval_cv                  1394
interval_min                 1189
seller_pro_share             1139
search_query_diversity       1061
interval_std                 1024
event_repeat_share           1009
chrome_major_mean             908
interval_mean                 858
item_view_diversity           810
session_duration_sec          797
interval_max                  779
photo_per_item_view           778
share_search_results_view     756
search_max_page               710
n_unique_locs                 691
mouse_x_std                   683


### submission

In [13]:
p_lgb = lgb_final.predict_proba(prep_lgb(df_test)[feat_cols])[:, 1]
p_cb  = cb_final.predict_proba(prep_cb(df_test)[feat_cols])[:, 1]
scores = blend_w * p_lgb + (1 - blend_w) * p_cb

sub = pd.DataFrame({"cookie_id": df_test["cookie_id"], "score": scores})
assert sub["cookie_id"].nunique() == len(sub)
assert sub["score"].between(0, 1).all()
sub.to_csv("submission.csv", index=False)